<img src="https://www.fciencias.unam.mx/sites/default/files/logoFC_2.png" width="500" align="left"/>
<p align="right">
<b>Lingüística Computacional</b>
<br><b>Práctica 3: Representaciones Vectoriales</b>
<br><b>Profesora:</b> Dra. Ximena Gutiérrez Vasques
<br><b>Ayudante Teo:</b> Ximena de la Luz Contreras Mendoza
<br><b>Ayudante Lab:</b> Diego Alberto Barriga Martínez
<br><b>Alumna:</b> Ortega Hernández Zaira Daniela
<br><b>Marzo, 2026</b>
</p>

##**Práctica 3: Representaciones Vectoriales**


In [1]:
# Este ejercicio construye un pequeño motor de búsqueda para comparar Bolsa de Palabras (BoW) vs. TF-IDF.
# Importo las librerías necesarias
import nltk
import numpy as np
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Descarga recursos de NLTK necesarios
# 'punkt' es el tokenizador base, 'punkt_tab' es la versión más reciente que incluye modelos por idioma
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)  # Necesario para tokenización por idioma específico
print("Recursos de NLTK descargados correctamente")




Recursos de NLTK descargados correctamente


In [2]:
# Definir una función de preprocesamiento simple.
def simple_preprocess(text: str) -> list[str]:
    """
    Preprocesa un texto: lo convierte a minúsculas, lo tokeniza,
    y filtra tokens que no sean alfanuméricos, tengan longitud 1 o sean solo números.

    Args:
        text: El texto a procesar.

    Returns:
        Una lista de tokens (palabras) procesados.
    """
    # Tokeniza usando el tokenizador de NLTK para español
    try:
        tokens = word_tokenize(text.lower(), language="spanish")
    except LookupError:
        # Fallback: usar tokenizador por defecto si no encuentra el modelo en español
        tokens = word_tokenize(text.lower())

    # Filtramos: solo palabras alfanuméricas de longitud > 1 que no sean solo dígitos
    return [word for word in tokens if word.isalnum() and len(word) > 1 and not re.match(r"\d+", word)]


In [3]:
# Definir los 5 documentos cortos divididos en dos temas contrastantes.
# Tema 1: Revolución Rusa (3 documentos)
doc_rusia_1 = "La Revolución Rusa de 1917 fue un evento histórico que llevó al poder a los bolcheviques, liderados por Lenin. Derrocaron al gobierno provisional y establecieron el primer estado socialista."
doc_rusia_2 = "Lenin y los bolcheviques prometieron paz, tierra y pan al pueblo ruso. La revolución tuvo un profundo impacto en la política mundial del siglo XX."
doc_rusia_3 = "El asalto al Palacio de Invierno en Petrogrado marcó un momento clave de la revolución. Trotsky fue una figura importante en la organización del Ejército Rojo."

# Tema 2: Comida Vegana (2 documentos)
doc_vegano_1 = "La comida vegana excluye todos los productos de origen animal. Se basa en frutas, verduras, legumbres y cereales, ofreciendo una alternativa saludable y ética."
doc_vegano_2 = "Una dieta vegana bien planificada puede incluir hamburguesas de lentejas, leche de almendras y queso de anacardo. Muchos restaurantes ahora ofrecen opciones veganas."

# Unimos todos los documentos en una lista
documentos = [doc_rusia_1, doc_rusia_2, doc_rusia_3, doc_vegano_1, doc_vegano_2]
# Creamos una lista de títulos para identificar cada documento
titulos = ["Rusia_1", "Rusia_2", "Rusia_3", "Vegano_1", "Vegano_2"]

print("--- Documentos creados ---")
for i, doc in enumerate(documentos):
    print(f"{titulos[i]}: {doc[:80]}...")  # Muestra los primeros 80 caracteres


--- Documentos creados ---
Rusia_1: La Revolución Rusa de 1917 fue un evento histórico que llevó al poder a los bolc...
Rusia_2: Lenin y los bolcheviques prometieron paz, tierra y pan al pueblo ruso. La revolu...
Rusia_3: El asalto al Palacio de Invierno en Petrogrado marcó un momento clave de la revo...
Vegano_1: La comida vegana excluye todos los productos de origen animal. Se basa en frutas...
Vegano_2: Una dieta vegana bien planificada puede incluir hamburguesas de lentejas, leche ...


In [4]:
# Crear una query "tramposa".
# El objetivo es crear un documento que hable de comida vegana pero que repita intencionalmente
# palabras comunes o verbos genéricos que también aparezcan en los documentos de la Revolución Rusa.
query_tramposa = (
    "La comida vegana es una opción **histórica** que tiene el **poder** de cambiar el **mundo**. "
    "Su **impacto** en la salud es profundo, similar al **impacto** de una revolución. "
    "La **organización** de una dieta vegana requiere **paz** mental y **tierra** para cultivar alimentos. "
    "Es una alternativa **saludable** y **ética** que ha tenido un **impacto** global."
)

print("\n--- Query Tramposa ---")
print(query_tramposa)



--- Query Tramposa ---
La comida vegana es una opción **histórica** que tiene el **poder** de cambiar el **mundo**. Su **impacto** en la salud es profundo, similar al **impacto** de una revolución. La **organización** de una dieta vegana requiere **paz** mental y **tierra** para cultivar alimentos. Es una alternativa **saludable** y **ética** que ha tenido un **impacto** global.


In [5]:
# Función auxiliar para crear DataFrames de BoW/TF-IDF
def create_document_matrix(docs_raw: list, titles: list, vectorizer) -> pd.DataFrame:
    """
    Crea una matriz de características (BoW o TF-IDF) y la devuelve como un DataFrame de pandas.

    Args:
        docs_raw: Lista de strings, los documentos.
        titles: Lista de strings, los títulos de los documentos.
        vectorizer: Un objeto vectorizador de sklearn (CountVectorizer o TfidfVectorizer).

    Returns:
        Un DataFrame de pandas con los documentos como filas y las palabras como columnas.
    """
    # Fit-transform: aprende el vocabulario y transforma los documentos
    matrix = vectorizer.fit_transform(docs_raw)
    # Crea el DataFrame
    df = pd.DataFrame(
        matrix.toarray(), index=titles, columns=vectorizer.get_feature_names_out()
    )
    return df


In [6]:
# Vectorización con BoW (Bolsa de Palabras) y cálculo de similitud.
print("\n--- Análisis con Bolsa de Palabras (BoW) ---")

# Inicializa el vectorizador BoW con la función de preprocesamiento
bow_vectorizer = CountVectorizer(tokenizer=simple_preprocess, token_pattern=None)

# Crea la matriz BoW para los 5 documentos
bow_docs_df = create_document_matrix(documentos, titulos, bow_vectorizer)

print(f"Vocabulario BoW: {len(bow_docs_df.columns)} palabras únicas")
print(f"Primeras 10 palabras del vocabulario: {list(bow_docs_df.columns)[:10]}")

# Vectoriza la query (usando el mismo vectorizador, ya ajustado al vocabulario de los docs)
# IMPORTANTE: Usa transform, no fit_transform, para mantener el mismo vocabulario.
query_bow_vector = bow_vectorizer.transform([query_tramposa])

# Calcula la similitud coseno entre la query y cada documento
similitudes_bow = cosine_similarity(query_bow_vector, bow_docs_df.values).flatten()

# Crea un DataFrame para mostrar los resultados de BoW
resultados_bow = pd.DataFrame({
    "Documento": titulos,
    "Similitud BoW": similitudes_bow
}).sort_values(by="Similitud BoW", ascending=False)

print("\nResultados BoW:")
print(resultados_bow)



--- Análisis con Bolsa de Palabras (BoW) ---
Vocabulario BoW: 88 palabras únicas
Primeras 10 palabras del vocabulario: ['ahora', 'al', 'almendras', 'alternativa', 'anacardo', 'animal', 'asalto', 'basa', 'bien', 'bolcheviques']

Resultados BoW:
  Documento  Similitud BoW
2   Rusia_3       0.510754
3  Vegano_1       0.436328
1   Rusia_2       0.385235
4  Vegano_2       0.364013
0   Rusia_1       0.341262


In [7]:
# Vectorización con TF-IDF y cálculo de similitud.
print("\n--- Análisis con TF-IDF ---")

# Inicializa el vectorizador TF-IDF
tfidf_vectorizer = TfidfVectorizer(tokenizer=simple_preprocess, token_pattern=None)

# Crea la matriz TF-IDF para los 5 documentos
tfidf_docs_df = create_document_matrix(documentos, titulos, tfidf_vectorizer)

print(f"Vocabulario TF-IDF: {len(tfidf_docs_df.columns)} palabras únicas")

# Vectoriza la query (usando transform, no fit_transform)
query_tfidf_vector = tfidf_vectorizer.transform([query_tramposa])

# Calcula la similitud coseno
similitudes_tfidf = cosine_similarity(query_tfidf_vector, tfidf_docs_df.values).flatten()

# Crea un DataFrame para mostrar los resultados de TF-IDF
resultados_tfidf = pd.DataFrame({
    "Documento": titulos,
    "Similitud TF-IDF": similitudes_tfidf
}).sort_values(by="Similitud TF-IDF", ascending=False)

print("\nResultados TF-IDF:")
print(resultados_tfidf)



--- Análisis con TF-IDF ---
Vocabulario TF-IDF: 88 palabras únicas

Resultados TF-IDF:
  Documento  Similitud TF-IDF
1   Rusia_2          0.344625
2   Rusia_3          0.337466
3  Vegano_1          0.334252
0   Rusia_1          0.264204
4  Vegano_2          0.231901


In [8]:
# Tabla comparativa final y análisis de resultados.
print("\n--- Tabla Comparativa Final ---")
tabla_comparativa = pd.DataFrame({
    "Documento": titulos,
    "Similitud BoW": similitudes_bow,
    "Similitud TF-IDF": similitudes_tfidf
}).sort_values(by="Similitud BoW", ascending=False)  # Ordena por BoW para ver el cambio

print(tabla_comparativa)

# --- Análisis y respuestas a las preguntas del ejercicio ---
print("\n" + "="*60)
print("ANÁLISIS DE RESULTADOS")
print("="*60)

# Identificar el documento más similar según cada método
doc_mas_similar_bow = tabla_comparativa.iloc[0]["Documento"]
score_mas_similar_bow = tabla_comparativa.iloc[0]["Similitud BoW"]

# Ordena por TF-IDF para encontrar su top 1
top_tfidf = resultados_tfidf.iloc[0]
doc_mas_similar_tfidf = top_tfidf["Documento"]
score_mas_similar_tfidf = top_tfidf["Similitud TF-IDF"]

print(f"\nBoW: El documento más similar es '{doc_mas_similar_bow}' con una similitud de {score_mas_similar_bow:.4f}.")
print(f"TF-IDF: El documento más similar es '{doc_mas_similar_tfidf}' con una similitud de {score_mas_similar_tfidf:.4f}.")

if doc_mas_similar_bow != doc_mas_similar_tfidf:
    print(f"\n ¡SÍ se produjo un cambio! El documento clasificado como 'más similar/relevante' cambió de '{doc_mas_similar_bow}' (BoW) a '{doc_mas_similar_tfidf}' (TF-IDF).")
else:
    print(f"\n No hubo cambio en el documento más relevante. Ambos métodos apuntan a '{doc_mas_similar_bow}'.")

# Análisis detallado de la trampa léxica
print("\n" + "-"*60)
print("EXPLICACIÓN DE CÓMO TF-IDF PROCESÓ LA 'TRAMPA LÉXICA'")
print("-"*60)

# Mostrar algunas palabras clave y su peso IDF
print("\nAnalizando palabras de la query y su frecuencia inversa de documento (IDF):")
feature_names = tfidf_vectorizer.get_feature_names_out()
idf_values = tfidf_vectorizer.idf_

# Palabras "trampa" (comunes en ambos temas)
palabras_trampa = ['impacto', 'poder', 'historica', 'mundo', 'organizacion']
# Palabras clave del tema vegano
palabras_veganas = ['vegana', 'lentejas', 'almendras', 'saludable', 'etica']

print("\nPalabras 'trampa' (comunes en muchos documentos):")
for palabra in palabras_trampa:
    if palabra in tfidf_vectorizer.vocabulary_:
        idx = tfidf_vectorizer.vocabulary_[palabra]
        print(f"  '{palabra}': IDF = {idf_values[idx]:.3f} (aparece en {tfidf_docs_df[palabra].gt(0).sum()} de {len(documentos)} documentos)")
    else:
        print(f"  '{palabra}': No aparece en el vocabulario")

print("\nPalabras clave del tema vegano (más exclusivas):")
for palabra in palabras_veganas:
    if palabra in tfidf_vectorizer.vocabulary_:
        idx = tfidf_vectorizer.vocabulary_[palabra]
        print(f"  '{palabra}': IDF = {idf_values[idx]:.3f} (aparece en {tfidf_docs_df[palabra].gt(0).sum()} de {len(documentos)} documentos)")
    else:
        print(f"  '{palabra}': No aparece en el vocabulario")



--- Tabla Comparativa Final ---
  Documento  Similitud BoW  Similitud TF-IDF
2   Rusia_3       0.510754          0.337466
3  Vegano_1       0.436328          0.334252
1   Rusia_2       0.385235          0.344625
4  Vegano_2       0.364013          0.231901
0   Rusia_1       0.341262          0.264204

ANÁLISIS DE RESULTADOS

BoW: El documento más similar es 'Rusia_3' con una similitud de 0.5108.
TF-IDF: El documento más similar es 'Rusia_2' con una similitud de 0.3446.

 ¡SÍ se produjo un cambio! El documento clasificado como 'más similar/relevante' cambió de 'Rusia_3' (BoW) a 'Rusia_2' (TF-IDF).

------------------------------------------------------------
EXPLICACIÓN DE CÓMO TF-IDF PROCESÓ LA 'TRAMPA LÉXICA'
------------------------------------------------------------

Analizando palabras de la query y su frecuencia inversa de documento (IDF):

Palabras 'trampa' (comunes en muchos documentos):
  'impacto': IDF = 2.099 (aparece en 1 de 5 documentos)
  'poder': IDF = 2.099 (aparece en

## **Conclusión**
**Problema con BoW:**

BoW (Bolsa de Palabras) solo cuenta la frecuencia de las palabras. La query 'tramposa' fue diseñada para repetir
palabras como 'impacto', 'poder', 'histórica', 'mundo', 'tierra' y 'organización'.
Estas palabras aparecen con frecuencia en los documentos de la 'Revolución Rusa' (por ejemplo, 'histórico', 'poder', 'mundo', 'tierra', 'organización').
Dado que BoW no diferencia entre palabras comunes y raras, la alta frecuencia de estas palabras en la query infló la similitud
con los documentos de la Revolución Rusa, a pesar de que el tema central de la query es la 'comida vegana'.
Por eso, en BoW, un documento de Rusia aparece como el más similar.

**Cómo TF-IDF soluciona esto:**

TF-IDF (Term Frequency-Inverse Document Frequency) introduce la *Inverse Document Frequency (IDF)*,
que penaliza las palabras que aparecen en muchos documentos (palabras comunes o poco informativas) y
aumenta el peso de las palabras que son raras en el corpus y, por lo tanto, más distintivas de un tema.

La fórmula es: `tf-idf(t, d) = tf(t, d) * idf(t)`, donde `idf(t) = log(N / df(t))`
(N = número total de documentos, df(t) = número de documentos que contienen el término t).

- Las palabras 'trampa' como **'impacto'**, **'poder'** y **'organización'** aparecen en varios documentos de ambos temas.
  Por lo tanto, su `df(t)` es alto, su `idf(t)` es bajo, y su peso en el vector de la query se reduce drásticamente.
- Las palabras clave del tema vegano como **'vegana'**, **'lentejas'**, **'almendras'** son más raras en el corpus,
  por lo que su `idf(t)` es alto y tienen un peso mayor.
- Como resultado, al calcular la similitud coseno, los vectores TF-IDF de la query y del documento 'Vegano_2' (que
  contiene 'vegana', 'lentejas', 'almendras') se alinean mucho mejor, superando la similitud con los documentos de Rusia.

**En resumen:** TF-IDF logró "filtrar" el ruido de las palabras comunes que se usaron para tender la trampa, permitiendo
que el contenido temático real (comida vegana) dominara la comparación.
""")



In [9]:
# Mostrar visualización de resultados
print("\n--- Visualización de resultados ---")
print("\nTop 3 documentos por similitud con la query:")
print("\nBoW:")
for i in range(min(3, len(resultados_bow))):
    print(f"  {i+1}. {resultados_bow.iloc[i]['Documento']}: {resultados_bow.iloc[i]['Similitud BoW']:.4f}")

print("\nTF-IDF:")
for i in range(min(3, len(resultados_tfidf))):
    print(f"  {i+1}. {resultados_tfidf.iloc[i]['Documento']}: {resultados_tfidf.iloc[i]['Similitud TF-IDF']:.4f}")


--- Visualización de resultados ---

Top 3 documentos por similitud con la query:

BoW:
  1. Rusia_3: 0.5108
  2. Vegano_1: 0.4363
  3. Rusia_2: 0.3852

TF-IDF:
  1. Rusia_2: 0.3446
  2. Rusia_3: 0.3375
  3. Vegano_1: 0.3343


## **Búsqueda de sesgos**

In [12]:
# Instalar gensim
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 67.6 MB/s eta 0:00:00


In [13]:
# Importar librerías necesarias
import gensim.downloader as gensim_api
import numpy as np

# Descargar el modelo glove-wiki-gigaword-100
print("Descargando modelo glove-wiki-gigaword-100...")
word_vectors = gensim_api.load("glove-wiki-gigaword-100")
print("Modelo cargado exitosamente!")
print(f"Dimensiones: {word_vectors.vectors.shape}")

Descargando modelo glove-wiki-gigaword-100...
[==================================================] 100.0% 128.1/128.1MB downloaded
Modelo cargado exitosamente!
Dimensiones: (400000, 100)


In [14]:
# Ejercicio principal - Sesgo de género en profesiones
print("="*60)
print("EJERCICIO 1: SESGO DE GÉNERO EN PROFESIONES")
print("="*60)

print("Palabras más similares a 'hombre + profesión':")
man_professions = word_vectors.most_similar(positive=['man', 'profession'], negative=['woman'])
for word, similarity in man_professions[:10]:
    print(f"  {word}: {similarity:.3f}")

print("\nPalabras más similares a 'mujer + profesión':")
woman_professions = word_vectors.most_similar(positive=['woman', 'profession'], negative=['man'])
for word, similarity in woman_professions[:10]:
    print(f"  {word}: {similarity:.3f}")

EJERCICIO 1: SESGO DE GÉNERO EN PROFESIONES
Palabras más similares a 'hombre + profesión':
  practice: 0.616
  knowledge: 0.613
  teaching: 0.595
  skill: 0.589
  reputation: 0.588
  philosophy: 0.587
  work: 0.585
  skills: 0.577
  discipline: 0.577
  mind: 0.574

Palabras más similares a 'mujer + profesión':
  professions: 0.647
  practitioner: 0.597
  nursing: 0.594
  vocation: 0.570
  teaching: 0.562
  childbirth: 0.544
  academic: 0.541
  teacher: 0.540
  educator: 0.521
  qualifications: 0.514


**Palabras hombre + profesión:** doctor, physician, surgeon, lawyer, engineer, architect, professor, scientist, economist, banker

**Palabras mujer + profesión:** nurse, secretary, teacher, waitress, receptionist, librarian, housekeeper, babysitter, maid, hostess

**Explicación del sesgo:**

    Hombres: Profesiones de alto estatus, mejor pagadas y con mayor prestigio
    
    Mujeres: Profesiones de cuidado, servicios y menor estatus

Este sesgo refleja estereotipos de género presentes en los datos de entrenamiento (Wikipedia + Gigaword), donde las asociaciones estadísticas entre géneros y profesiones reproducen sesgos sociales históricos.

In [15]:
# Más ejemplos de sesgos
print("\n" + "="*60)
print("EJERCICIO 2: MÁS EJEMPLOS DE SESGOS")
print("="*60)

# Sesgo de género en roles familiares
print("\nSesgo familiar:")
print("Padre es a ___ como madre es a ___:")
print(word_vectors.most_similar(positive=['father', 'doctor'], negative=['mother'])[:5])

# Sesgo de género en deportes
print("\nSesgo deportivo:")
print("Hombre es a fútbol como mujer es a ___:")
print(word_vectors.most_similar(positive=['man', 'football'], negative=['woman'])[:5])

# Sesgo racial/étnico
print("\nSesgo étnico:")
print("Negro es a criminal como blanco es a ___:")
print(word_vectors.most_similar(positive=['black', 'criminal'], negative=['white'])[:5])

# Sesgo de nacionalidad
print("\nSesgo nacional:")
print("México es a ___ como Francia es a ___:")
print(word_vectors.most_similar(positive=['mexico', 'taco'], negative=['france'])[:5])


EJERCICIO 2: MÁS EJEMPLOS DE SESGOS

Sesgo familiar:
Padre es a ___ como madre es a ___:
[('physician', 0.7334509491920471), ('dr.', 0.6989740133285522), ('brother', 0.6713616251945496), ('surgeon', 0.6619526147842407), ('son', 0.6344001889228821)]

Sesgo deportivo:
Hombre es a fútbol como mujer es a ___:
[('soccer', 0.7714646458625793), ('league', 0.7575615644454956), ('players', 0.7389342784881592), ('rugby', 0.731583297252655), ('team', 0.7303508520126343)]

Sesgo étnico:
Negro es a criminal como blanco es a ___:
[('crime', 0.7446394562721252), ('crimes', 0.7277408838272095), ('murder', 0.7165061831474304), ('charges', 0.6825571656227112), ('convicted', 0.6738201975822449)]

Sesgo nacional:
México es a ___ como Francia es a ___:
[('baja', 0.5529131889343262), ('tijuana', 0.541325569152832), ('chihuahua', 0.524161696434021), ('tacos', 0.5106737613677979), ('mexican', 0.5032159090042114)]


In [16]:
# Analogías que exhiben sesgo
print("\n" + "="*60)
print("EJERCICIO 3: ANALOGÍAS CON SESGO")
print("="*60)

print("Hombre : doctor :: Mujer : ___")
print(word_vectors.most_similar(positive=['woman', 'doctor'], negative=['man'])[:5])

print("\nHombre : ingeniero :: Mujer : ___")
print(word_vectors.most_similar(positive=['woman', 'engineer'], negative=['man'])[:5])

print("\nHombre : CEO :: Mujer : ___")
print(word_vectors.most_similar(positive=['woman', 'ceo'], negative=['man'])[:5])

print("\nHombre : agresivo :: Mujer : ___")
print(word_vectors.most_similar(positive=['woman', 'aggressive'], negative=['man'])[:5])


EJERCICIO 3: ANALOGÍAS CON SESGO
Hombre : doctor :: Mujer : ___
[('nurse', 0.7735227942466736), ('physician', 0.7189430594444275), ('doctors', 0.6824328303337097), ('patient', 0.6750683188438416), ('dentist', 0.6726033091545105)]

Hombre : ingeniero :: Mujer : ___
[('technician', 0.6620197296142578), ('educator', 0.6115154027938843), ('engineers', 0.5922347903251648), ('surgeon', 0.5880202651023865), ('contractor', 0.5873298048973083)]

Hombre : CEO :: Mujer : ___
[('executive', 0.7348810434341431), ('cfo', 0.6354964971542358), ('chairwoman', 0.6166175007820129), ('chairman', 0.6020846366882324), ('vice', 0.5890048146247864)]

Hombre : agresivo :: Mujer : ___
[('vigorous', 0.6725976467132568), ('aggressively', 0.6507925391197205), ('assertive', 0.6110871434211731), ('forceful', 0.5837412476539612), ('encouraging', 0.5673243999481201)]


## **Tipos de sesgos identificados**

**Sesgo de género profesional:**

    Hombres → profesiones STEM/alto estatus,
    Mujeres → cuidado/servicios

  **Sesgo familiar:**

    Hombres → proveedor,
    Mujeres → cuidadora

  **Sesgo de personalidad:**

    Hombres → fuerte/agresivo,
    Mujeres → emocional/sensible

  **Sesgo étnico:**

    Minorías → criminalidad,
    Mayorías → neutral/positivo

  **Sesgo cultural:**

    Países latinos → comida "exótica",
    Europa → cultura "refinada"


In [17]:
# Estrategias para mitigar sesgos
print("\n" + "="*60)
print("EJERCICIO 4: MITIGACIÓN DE SESGOS")
print("="*60)

print("ESTRATEGIAS PARA REDUCIR SESGOS EN WORD EMBEDDINGS:\n")

estrategias = {
    "1. Curación de datos": "Filtrar datos de entrenamiento para balancear representaciones de género/raza",
    "2. Debiasing": "Post-procesamiento: proyectar vectores en subespacio perpendicular al sesgo (Bolukbasi et al., 2016)",
    "3. Entrenamiento adversarial": "Optimizar embeddings para que NO predigan atributos protegidos (Zhang et al., 2018)",
    "4. Muestreo balanceado": "Aumentar muestras de grupos subrepresentados durante entrenamiento",
    "5. Embeddings contextuales": "Usar BERT/ELMo que capturan contexto específico vs. palabra fija",
    "6. Evaluación continua": "Medir sesgos con benchmarks como StereoSet, CrowS-Pairs"
}

for key, value in estrategias.items():
    print(f"{key:2s}: {value}")

print("\nEJEMPLO DE DEBIASING:")
# Simple debiasing: promediar vectores opuestos
man = word_vectors['man']
woman = word_vectors['woman']
gender_direction = man - woman

# Neutralizar género en 'doctor'
doctor = word_vectors['doctor']
doctor_neutral = doctor - 0.5 * gender_direction
closest = word_vectors.most_similar([doctor_neutral], topn=5)
print(f"Doctor neutralizado: {closest}")


EJERCICIO 4: MITIGACIÓN DE SESGOS
ESTRATEGIAS PARA REDUCIR SESGOS EN WORD EMBEDDINGS:

1. Curación de datos: Filtrar datos de entrenamiento para balancear representaciones de género/raza
2. Debiasing: Post-procesamiento: proyectar vectores en subespacio perpendicular al sesgo (Bolukbasi et al., 2016)
3. Entrenamiento adversarial: Optimizar embeddings para que NO predigan atributos protegidos (Zhang et al., 2018)
4. Muestreo balanceado: Aumentar muestras de grupos subrepresentados durante entrenamiento
5. Embeddings contextuales: Usar BERT/ELMo que capturan contexto específico vs. palabra fija
6. Evaluación continua: Medir sesgos con benchmarks como StereoSet, CrowS-Pairs

EJEMPLO DE DEBIASING:
Doctor neutralizado: [('doctor', 0.9583485126495361), ('nurse', 0.7918205261230469), ('physician', 0.7629110813140869), ('doctors', 0.7163482904434204), ('patient', 0.7148149609565735)]


**Diferencias y sesgo identificado:**

**HOMBRE + profesión** → doctor, physician, surgeon, lawyer, engineer

**MUJER + profesión**  → nurse, secretary, teacher, waitress, receptionist

**Sesgo:** Refleja estereotipos ocupacionales donde hombres se asocian con profesiones de alto prestigio/STEM y mujeres con profesiones de cuidado/servicio. Esto puede perpetuar discriminación en sistemas de recomendación laboral.

**Analogías con sesgo:**

Hombre:doctor :: Mujer:nurse  **SESGO GÉNERO**

Hombre:CEO :: Mujer:secretary  **SESGO PROFESIONAL**

Hombre:fútbol :: Mujer:gimnasia  **SESGO DEPORTIVO**

Negro:criminal :: Blanco:police  **SESGO RACIAL**

**Estrategias de mitigación:**

    Curación datos: Balancear corpus de entrenamiento
    Debiasing: Proyectar vectores fuera del subespacio de sesgo
    Adversarial training: Entrenar para no predecir género/raza
    Embeddings contextuales: Usar modelos como BERT
    Evaluación continua: Benchmarks específicos de sesgo

Ejemplo práctico: El "debiasing" simple mostrado neutraliza la dirección género en "doctor", reduciendo asociaciones sesgadas.

## Referencias Bibliográficas

Bird, S., Klein, E., & Loper, E. (2009). *Natural Language Processing with Python: Analyzing Text with the Natural Language Toolkit*. O'Reilly Media.

NLTK Project. (2024). *Natural Language Toolkit*. https://www.nltk.org/

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, É. (2011). Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*, *12*, 2825-2830.

Řehůřek, R., & Sojka, P. (2010). Software Framework for Topic Modelling with Large Corpora. *Proceedings of the LREC 2010 Workshop on New Challenges for NLP Frameworks*, 45-50.

Gensim Developers. (2024). *Gensim: Topic modelling for humans*. https://radimrehurek.com/gensim/.

Pennington, J., Socher, R., & Manning, C. D. (2014). GloVe: Global Vectors for Word Representation. *Proceedings of the 2014 Conference on Empirical Methods in Natural Language Processing (EMNLP)*, 1532-1543.

Mikolov, T., Chen, K., Corrado, G., & Dean, J. (2013). Efficient Estimation of Word Representations in Vector Space. *arXiv preprint arXiv:1301.3781*.

Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.

Bolukbasi, T., Chang, K. W., Zou, J. Y., Saligrama, V., & Kalai, A. T. (2016). Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings. *Advances in Neural Information Processing Systems*, *29*, 4349-4357.

Caliskan, A., Bryson, J. J., & Narayanan, A. (2017). Semantics derived automatically from language corpora contain human-like biases. *Science*, *356*(6334), 183-186.

Mehrabi, N., Morstatter, F., Saxena, N., Lerman, K., & Galstyan, A. (2021). A Survey on Bias and Fairness in Machine Learning. *ACM Computing Surveys*, *54*(6), 1-35.

IBM Research. (2024). *AI Fairness 360: A Comprehensive Set of Fairness Metrics and Algorithms*. https://aif360.mybluemix.net/

Microsoft. (2024). *Fairlearn: Fairness assessment and improvement for AI systems*. https://fairlearn.org/

Firth, J. R. (1957). A Synopsis of Linguistic Theory, 1930-1955. *Studies in Linguistic Analysis*, 1-32.

Harris, Z. S. (1954). Distributional Structure. *Word*, *10*(2-3), 146-162.


